# 4. LangChain の基礎


In [1]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

## 4.1. LangChain の概要


### LangChain のインストール


#### 【注意】既知のエラーについて

pydantic のアップデートにより、明示的に pydantic のバージョンを指定していない箇所で ChatOpenAI などを使用すると、`PydanticUserError: 'ChatOpenAI' is not fully defined; you should define 'BaseCache', then call 'ChatOpenAI.model_rebuild()'.` というエラーが発生するようになりました。

このエラーは、`!pip install pydantic==2.10.6` のように、pydantic の特定バージョンをインストールすることで回避することができます。

なお、Google Colab で一度上記のエラーに遭遇したあとで `!pip install pydantic==2.10.6` のようにパッケージをインストールし直した場合、以下のどちらかの操作を実施する必要があります。

- Google Colab の「ランタイム」から「セッションを再起動する」を実行する
- 「ランタイムを接続解除して削除」を実行してパッケージのインストールからやり直す


In [ ]:
!pip install langchain-core==0.3.0 langchain-openai==0.2.0 pydantic==2.10.6

### LangSmith のセットアップ


In [2]:
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_API_KEY"] = userdata.get("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_PROJECT"] = "agent-book"

## 4.2. LLM / Chat model


### LLM


In [ ]:
from langchain_openai import OpenAI

model = OpenAI(model="gpt-3.5-turbo-instruct", temperature=0)
ai_message = model.invoke("こんにちは")
print(ai_message)

### Chat model


In [ ]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

messages = [
    SystemMessage("You are a helpful assistant."),
    HumanMessage("こんにちは！私はジョンと言います"),
    AIMessage(content="こんにちは、ジョンさん！どのようにお手伝いできますか？"),
    HumanMessage(content="私の名前がわかりますか？"),
]

ai_message = model.invoke(messages)
print(ai_message.content)

### ストリーミング


In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

messages = [
    SystemMessage("You are a helpful assistant."),
    HumanMessage("こんにちは！"),
]

for chunk in model.stream(messages):
    print(chunk.content, end="", flush=True)

## 4.3. Prompt template


### PromptTemplate


In [ ]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template("""以下の料理のレシピを考えてください。

料理名: {dish}""")

prompt_value = prompt.invoke({"dish": "カレー"})
print(prompt_value.text)

#### ＜補足：プロンプトの変数が 1 つの場合＞


In [ ]:
prompt_value = prompt.invoke("カレー")
print(prompt_value.text)

### ChatPromptTemplate


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "ユーザーが入力した料理のレシピを考えてください。"),
        ("human", "{dish}"),
    ]
)

prompt_value = prompt.invoke({"dish": "カレー"})
print(prompt_value)

### MessagesPlaceholder


In [ ]:
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant."),
        MessagesPlaceholder("chat_history", optional=True),
        ("human", "{input}"),
    ]
)

prompt_value = prompt.invoke(
    {
        "chat_history": [
            HumanMessage(content="こんにちは！私はジョンと言います！"),
            AIMessage("こんにちは、ジョンさん！どのようにお手伝いできますか？"),
        ],
        "input": "私の名前が分かりますか？",
    }
)
print(prompt_value)

### LangSmith の Prompts


In [ ]:
from langsmith import Client

client = Client()
prompt = client.pull_prompt("oshima/recipe")

prompt_value = prompt.invoke({"dish": "カレー"})
print(prompt_value)

### （コラム）マルチモーダルモデルの入力の扱い


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "user",
            [
                {"type": "text", "text": "画像を説明してください。"},
                {"type": "image_url", "image_url": {"url": "{image_url}"}},
            ],
        ),
    ]
)
image_url = "https://raw.githubusercontent.com/yoshidashingo/langchain-book/main/assets/cover.jpg"

prompt_value = prompt.invoke({"image_url": image_url})

In [ ]:
model = ChatOpenAI(model="gpt-4o", temperature=0)
ai_message = model.invoke(prompt_value)
print(ai_message.content)

## 4.4. Output parser


### PydanticOutputParser を使った Python オブジェクトへの変換


In [ ]:
from pydantic import BaseModel, Field


class Recipe(BaseModel):
    ingredients: list[str] = Field(description="ingredients of the dish")
    steps: list[str] = Field(description="steps to make the dish")

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser

output_parser = PydanticOutputParser(pydantic_object=Recipe)

In [ ]:
format_instructions = output_parser.get_format_instructions()
print(format_instructions)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "ユーザーが入力した料理のレシピを考えてください。\n\n"
            "{format_instructions}",
        ),
        ("human", "{dish}"),
    ]
)

prompt_with_format_instructions = prompt.partial(
    format_instructions=format_instructions
)

In [ ]:
prompt_value = prompt_with_format_instructions.invoke({"dish": "カレー"})
print("=== role: system ===")
print(prompt_value.messages[0].content)
print("=== role: user ===")
print(prompt_value.messages[1].content)

In [ ]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

ai_message = model.invoke(prompt_value)
print(ai_message.content)

In [ ]:
recipe = output_parser.invoke(ai_message)
print(type(recipe))
print(recipe)

### StrOutputParser


In [ ]:
from langchain_core.messages import AIMessage
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()

ai_message = AIMessage(content="こんにちは。私はAIアシスタントです。")
ai_message = output_parser.invoke(ai_message)
print(type(ai_message))
print(ai_message)

## 4.5.Chain―LangChain Expression Language（LCEL）の概要


### prompt と model の連鎖


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "ユーザーが入力した料理のレシピを考えてください。"),
        ("human", "{dish}"),
    ]
)

model = ChatOpenAI(model_name="gpt-4o-mini", temperature=0)

In [ ]:
chain = prompt | model

In [ ]:
ai_message = chain.invoke({"dish": "カレー"})
print(ai_message.content)

### StrOutputParser を連鎖に追加


In [ ]:
from langchain_core.output_parsers import StrOutputParser

chain = prompt | model | StrOutputParser()
output = chain.invoke({"dish": "カレー"})
print(output)

### PydanticOutputParser を使う連鎖


In [ ]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field


class Recipe(BaseModel):
    ingredients: list[str] = Field(description="ingredients of the dish")
    steps: list[str] = Field(description="steps to make the dish")


output_parser = PydanticOutputParser(pydantic_object=Recipe)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "ユーザーが入力した料理のレシピを考えてください。\n\n{format_instructions}"),
        ("human", "{dish}"),
    ]
)

prompt_with_format_instructions = prompt.partial(
    format_instructions=output_parser.get_format_instructions()
)

model = ChatOpenAI(model="gpt-4o-mini", temperature=0).bind(
    response_format={"type": "json_object"}
)

In [ ]:
chain = prompt_with_format_instructions | model | output_parser

In [ ]:
recipe = chain.invoke({"dish": "カレー"})
print(type(recipe))
print(recipe)

### （コラム）with_structured_output


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field


class Recipe(BaseModel):
    ingredients: list[str] = Field(description="ingredients of the dish")
    steps: list[str] = Field(description="steps to make the dish")


prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "ユーザーが入力した料理のレシピを考えてください。"),
        ("human", "{dish}"),
    ]
)

model = ChatOpenAI(model="gpt-4o-mini")

chain = prompt | model.with_structured_output(Recipe)

recipe = chain.invoke({"dish": "カレー"})
print(type(recipe))
print(recipe)

## 4.6.LangChain の RAG に関するコンポーネント


### Document loader


In [ ]:
!pip install langchain-community==0.3.0 GitPython==3.1.43

In [ ]:
from langchain_community.document_loaders import GitLoader


def file_filter(file_path: str) -> bool:
    return file_path.endswith(".mdx")


loader = GitLoader(
    clone_url="https://github.com/langchain-ai/langchain",
    repo_path="./langchain",
    branch="master",
    file_filter=file_filter,
)

raw_docs = loader.load()
print(len(raw_docs))

### Document transformer


In [ ]:
!pip install langchain-text-splitters==0.3.0

In [ ]:
from langchain_text_splitters import CharacterTextSplitter

text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)

docs = text_splitter.split_documents(raw_docs)
print(len(docs))

### Embedding model


In [ ]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [ ]:
query = "AWSのS3からデータを読み込むためのDocument loaderはありますか？"

vector = embeddings.embed_query(query)
print(len(vector))
print(vector)

### Vector store


In [2]:
!pip install langchain-chroma==0.1.4

ERROR: Could not find a version that satisfies the requirement langchain_community.vectorstores (from versions: none)
ERROR: No matching distribution found for langchain_community.vectorstores


In [ ]:
from langchain_chroma import Chroma

db = Chroma.from_documents(docs, embeddings)

In [ ]:
retriever = db.as_retriever()

In [ ]:
query = "AWSのS3からデータを読み込むためのDocument loaderはありますか？"

context_docs = retriever.invoke(query)
print(f"len = {len(context_docs)}")

first_doc = context_docs[0]
print(f"metadata = {first_doc.metadata}")
print(first_doc.page_content)

### LCEL を使った RAG の Chain の実装


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

prompt = ChatPromptTemplate.from_template('''\
以下の文脈だけを踏まえて質問に回答してください。

文脈: """
{context}
"""

質問: {question}
''')

model = ChatOpenAI(model_name="gpt-4o-mini", temperature=0)

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

output = chain.invoke(query)
print(output)

In [16]:
import os,time,re,requests
from collections import deque
from urllib.parse import urljoin,urldefrag,urlparse

from bs4 import BeautifulSoup
from langchain_core.documents import Document
from langchain_openai import OpenAI,OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

BASE_URL="https://marinediving.com/area/"
question="初心者向けでサンゴが綺麗な海は？"

MAX_DEPTH=2
MAX_PAGES=60
SLEEP=0.2
TOP_K=3
MAX_CTX_CHARS=4500

basep=urlparse(BASE_URL)
BASE_HOST=basep.netloc

def in_scope(u: str) -> bool:
    x=urlparse(u)
    if x.netloc != BASE_HOST:
        return False
    # /area/配下だけを対象（末尾スラッシュ/ html 両対応）
    return x.path.startswith("/area/")

def norm(base,href):
    if not href: return None
    u,_=urldefrag(urljoin(base,href))
    return u if in_scope(u) else None

def fetch(url):
    r=requests.get(url,timeout=20,headers={"User-Agent":"tasamu-colab/1.0"})
    r.encoding = r.apparent_encoding or r.encoding
    return r.text

def parse(url,html):
    soup=BeautifulSoup(html,"html.parser")
    # /area/配下リンク抽出
    links=[norm(url,a.get("href")) for a in soup.select("a[href]")]
    links=[u for u in links if u]
    # 本文化
    for t in soup(["script","style","noscript"]): t.decompose()
    text=re.sub(r"\n{3,}","\n\n",soup.get_text("\n")).strip()
    return text,links

def crawl():
    q=deque([(BASE_URL,0)])
    seen=set()
    docs=[]
    while q and len(seen)<MAX_PAGES:
        url,depth=q.popleft()
        if url in seen:
            continue
        seen.add(url)
        try:
            html=fetch(url)
            text,links=parse(url,html)

            # ログ：辿れるリンクが見えているか確認
            if url == BASE_URL:
                print("seed links found:", len(links))
                print("seed sample:", links[:20])

            if len(text)>=300:
                docs.append(Document(page_content=text,metadata={"source":url}))

            if depth<MAX_DEPTH:
                for u in links:
                    if u not in seen:
                        q.append((u,depth+1))
        except:
            pass
        time.sleep(SLEEP)
    return docs,len(seen)

def unique_by_source(docs,limit=TOP_K):
    out=[]; s=set()
    for d in docs:
        src=d.metadata.get("source")
        if src in s:
            continue
        s.add(src); out.append(d)
        if len(out)>=limit:
            break
    return out

def format_docs_limited(docs):
    t="\n\n".join(d.page_content for d in docs)
    return t[:MAX_CTX_CHARS]

if not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY を設定してください")

docs,visited=crawl()
print(f"visited urls: {visited}, loaded docs: {len(docs)}")

splits=RecursiveCharacterTextSplitter(chunk_size=400,chunk_overlap=50).split_documents(docs)
db=FAISS.from_documents(splits,OpenAIEmbeddings())
retriever=db.as_retriever(search_kwargs={"k":6})

prompt=PromptTemplate.from_template(
"次の情報だけを根拠に質問に答えてください。\n"
"回答は日本語で200文字以下。\n"
"根拠にできる地名やポイント名があれば含めてください。\n"
"情報に無い内容は推測せず「不明」と答えてください。\n\n"
"情報:\n{context}\n\n質問:\n{question}\n\n回答:"
)

llm=OpenAI(model="gpt-3.5-turbo-instruct",temperature=0)

chain=(
 {"context": (retriever
              | (lambda ds: unique_by_source(ds,limit=TOP_K))
              | format_docs_limited),
  "question": RunnablePassthrough()}
 | prompt | llm | StrOutputParser()
)

answer=chain.invoke(question).strip()
print("Q:",question)
print("A:",answer)

sources=unique_by_source(retriever.invoke(question),limit=TOP_K)
print("\n--- sources ---")
for i,sd in enumerate(sources,1):
    print(f"[{i}] {sd.metadata.get('source')}")


seed links found: 309
seed sample: ['https://marinediving.com/area/', 'https://marinediving.com/area/', 'https://marinediving.com/area/', 'https://marinediving.com/area/', 'https://marinediving.com/area/', 'https://marinediving.com/area/', 'https://marinediving.com/area/', 'https://marinediving.com/area/', 'https://marinediving.com/area/', 'https://marinediving.com/area/', 'https://marinediving.com/area/', 'https://marinediving.com/area/', 'https://marinediving.com/area/', 'https://marinediving.com/area/', 'https://marinediving.com/area/', 'https://marinediving.com/area/', 'https://marinediving.com/area/', 'https://marinediving.com/area/', 'https://marinediving.com/area/', 'https://marinediving.com/area/']
visited urls: 60, loaded docs: 60
Q: 初心者向けでサンゴが綺麗な海は？
A: 西表島の中でも特に初心者向けで、美しいサンゴが広がる海は、浅瀬の根のトップにこれでもか！とサンゴが群生するイジャカジャ（伊釈迦釈）や、浅い砂地に何百ものガーデンイールやヤシャハゼ、イソバナにはキンメモドキやスカシテンジクダイなどが舞う写真／宮城清（ダイビングチームあなたの清）が挙げられます。また、石垣港から5分の距離にある港から5分の楽園や、嘉弥真島へ向かう途中にある三つ石も初心者におすすめのスポットです。

--- sources ---
[1] 

In [19]:
def ask(q):
    a = chain.invoke(q).strip().replace("\n", " ")
    return a[:200]  # 200文字以下に確実化

print(ask("海外で大物が出る上級者ポイントは？"))


情報には海外の情報は含まれていないため、推測はできません。


In [15]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

BASE_URL = "https://marinediving.com/area/"

r = requests.get(BASE_URL, timeout=15, headers={"User-Agent":"tasamu-colab/1.0"})
r.encoding = r.apparent_encoding  # 自動推定
soup = BeautifulSoup(r.text, "html.parser")

links = [urljoin(BASE_URL, a["href"]) for a in soup.select("a[href]")]
print("a[href] count:", len(links))
print("sample:", links[:30])




a[href] count: 466
sample: ['https://marinediving.com/', 'https://marinediving.com/area/', 'https://marinediving.com/feature/', 'https://marinediving.com/marinediving_awards/', 'https://marinediving.com/photography/professional/', 'https://marinediving.com/skill/scubaproshop/50/', 'https://marinediving.com/photography/', 'https://marinediving.com/safety_diving/stop_accidents/case147/', 'https://marinediving.com/safety_diving/#a2', 'https://marinediving.com/start_diving/license/', 'https://marinediving.com/environment/', 'https://marinediving.com/area/', 'https://marinediving.com/mdf/', 'https://marinediving.com/topics/32292.html', 'https://marinediving.com/mdf/about/', 'https://marinediving.com/mdf/zone/', 'https://marinediving.com/mdf/report2025/', 'https://marinediving.com/mdf/brochure/', 'https://marinediving.com/mdf/stage/', 'https://marinediving.com/mdf/present/', 'https://marinediving.com/topics/', 'https://marinediving.com/area/', 'https://marinediving.com/area/#a2', 'https://ma

# build_index.py
import os
import re
import time
import json
import hashlib
import sqlite3
from collections import deque
from urllib.parse import urljoin, urldefrag, urlparse

import requests
from bs4 import BeautifulSoup

from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter


# ===== 設定 =====
BASE_URL = "https://marinediving.com/area/"   # 対象サイト
MAX_DEPTH = 2
MAX_PAGES = 300
SLEEP_SEC = 0.2
TIMEOUT_SEC = 20

# 文字コード：基本は自動推定。SJIS固定したい場合は "shift_jis" を入れる
FORCE_ENCODING = None  # 例: "shift_jis"

# 保存先
DATA_DIR = "./rag_data"
RAW_DIR = os.path.join(DATA_DIR, "raw")
CLEAN_DIR = os.path.join(DATA_DIR, "clean")
DB_PATH = os.path.join(DATA_DIR, "meta.sqlite3")
INDEX_DIR = os.path.join(DATA_DIR, "index_faiss")

# チャンク
CHUNK_SIZE = 400
CHUNK_OVERLAP = 50


def ensure_dirs():
    os.makedirs(RAW_DIR, exist_ok=True)
    os.makedirs(CLEAN_DIR, exist_ok=True)
    os.makedirs(DATA_DIR, exist_ok=True)


def url_key(url: str) -> str:
    return hashlib.sha256(url.encode("utf-8")).hexdigest()


def in_scope(url: str, base_host: str, base_path_prefix: str) -> bool:
    u = urlparse(url)
    return (u.netloc == base_host) and u.path.startswith(base_path_prefix)


def norm_url(base: str, href: str, base_host: str, base_path_prefix: str) -> str | None:
    if not href:
        return None
    u, _ = urldefrag(urljoin(base, href))
    return u if in_scope(u, base_host, base_path_prefix) else None


def fetch_html(url: str) -> str:
    r = requests.get(url, timeout=TIMEOUT_SEC, headers={"User-Agent": "tasamu-rag/1.0"})
    if FORCE_ENCODING:
        r.encoding = FORCE_ENCODING
    else:
        r.encoding = r.apparent_encoding or r.encoding
    return r.text


def html_to_text_and_links(url: str, html: str, base_host: str, base_path_prefix: str):
    soup = BeautifulSoup(html, "html.parser")

    links = []
    for a in soup.select("a[href]"):
        nxt = norm_url(url, a.get("href"), base_host, base_path_prefix)
        if nxt:
            links.append(nxt)

    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    text = soup.get_text(separator="\n")
    text = re.sub(r"\n{3,}", "\n\n", text).strip()
    return text, links


def init_db():
    con = sqlite3.connect(DB_PATH)
    cur = con.cursor()
    cur.execute("""
        CREATE TABLE IF NOT EXISTS pages (
            url TEXT PRIMARY KEY,
            url_key TEXT,
            fetched_at INTEGER,
            content_hash TEXT,
            raw_path TEXT,
            clean_path TEXT,
            status TEXT
        )
    """)
    con.commit()
    return con


def get_page_record(con, url: str):
    cur = con.cursor()
    cur.execute("SELECT content_hash, clean_path FROM pages WHERE url=?", (url,))
    row = cur.fetchone()
    return row  # (content_hash, clean_path) or None


def upsert_page_record(con, url: str, content_hash: str, raw_path: str, clean_path: str, status: str):
    cur = con.cursor()
    cur.execute("""
        INSERT INTO pages(url, url_key, fetched_at, content_hash, raw_path, clean_path, status)
        VALUES(?,?,?,?,?,?,?)
        ON CONFLICT(url) DO UPDATE SET
            fetched_at=excluded.fetched_at,
            content_hash=excluded.content_hash,
            raw_path=excluded.raw_path,
            clean_path=excluded.clean_path,
            status=excluded.status
    """, (url, url_key(url), int(time.time()), content_hash, raw_path, clean_path, status))
    con.commit()


def write_text(path: str, text: str):
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)


def write_raw(path: str, html: str):
    with open(path, "w", encoding="utf-8", errors="ignore") as f:
        f.write(html)


def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def crawl_and_save():
    ensure_dirs()
    con = init_db()

    basep = urlparse(BASE_URL)
    base_host = basep.netloc
    base_path_prefix = basep.path if basep.path.endswith("/") else basep.path + "/"

    q = deque([(BASE_URL, 0)])
    seen = set()
    saved_docs = []

    while q and len(seen) < MAX_PAGES:
        url, depth = q.popleft()
        if url in seen:
            continue
        seen.add(url)

        try:
            html = fetch_html(url)
            text, links = html_to_text_and_links(url, html, base_host, base_path_prefix)

            # 次のURLをキューへ
            if depth < MAX_DEPTH:
                for nxt in links:
                    if nxt not in seen:
                        q.append((nxt, depth + 1))

            # 本文が薄すぎるページは保存しない（必要なら閾値調整）
            if len(text) < 300:
                upsert_page_record(con, url, "", "", "", "skipped_thin")
                time.sleep(SLEEP_SEC)
                continue

            h = sha256_text(text)
            rec = get_page_record(con, url)
            if rec and rec[0] == h:
                # 変更なし：既存cleanを使う
                clean_path = rec[1]
                if clean_path and os.path.exists(clean_path):
                    with open(clean_path, "r", encoding="utf-8") as f:
                        saved_docs.append(Document(page_content=f.read(), metadata={"source": url}))
                upsert_page_record(con, url, h, "", clean_path, "unchanged")
                time.sleep(SLEEP_SEC)
                continue

            # 変更あり/新規：保存
            k = url_key(url)
            raw_path = os.path.join(RAW_DIR, f"{k}.html")
            clean_path = os.path.join(CLEAN_DIR, f"{k}.txt")

            write_raw(raw_path, html)
            write_text(clean_path, text)
            upsert_page_record(con, url, h, raw_path, clean_path, "saved")

            saved_docs.append(Document(page_content=text, metadata={"source": url}))

        except Exception:
            upsert_page_record(con, url, "", "", "", "fetch_failed")

        time.sleep(SLEEP_SEC)

    con.close()
    return saved_docs, len(seen)


def build_faiss(docs):
    splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
    splits = splitter.split_documents(docs)

    db = FAISS.from_documents(splits, OpenAIEmbeddings())
    db.save_local(INDEX_DIR)

    # 参考情報を保存
    meta = {
        "base_url": BASE_URL,
        "chunk_size": CHUNK_SIZE,
        "chunk_overlap": CHUNK_OVERLAP,
        "created_at": int(time.time()),
        "doc_count": len(docs),
        "chunk_count": len(splits),
    }
    with open(os.path.join(DATA_DIR, "index_meta.json"), "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)


def main():
    if not os.environ.get("OPENAI_API_KEY"):
        raise RuntimeError("OPENAI_API_KEY を設定してください")

    docs, visited = crawl_and_save()
    print(f"visited urls: {visited}, loaded docs: {len(docs)}")

    if not docs:
        raise RuntimeError("有効な本文が取得できませんでした。")

    build_faiss(docs)
    print(f"saved index: {INDEX_DIR}")


if __name__ == "__main__":
    main()


In [ ]:
# build_index.py
import os
import re
import time
import json
import hashlib
import sqlite3
from collections import deque
from urllib.parse import urljoin, urldefrag, urlparse

import requests
from bs4 import BeautifulSoup

from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter


# ===== 設定 =====
BASE_URL = "https://marinediving.com/area/"   # 対象サイト
MAX_DEPTH = 2
MAX_PAGES = 300
SLEEP_SEC = 0.2
TIMEOUT_SEC = 20

# 文字コード：基本は自動推定。SJIS固定したい場合は "shift_jis" を入れる
FORCE_ENCODING = None  # 例: "shift_jis"

# 保存先
DATA_DIR = "./rag_data"
RAW_DIR = os.path.join(DATA_DIR, "raw")
CLEAN_DIR = os.path.join(DATA_DIR, "clean")
DB_PATH = os.path.join(DATA_DIR, "meta.sqlite3")
INDEX_DIR = os.path.join(DATA_DIR, "index_faiss")

# チャンク
CHUNK_SIZE = 400
CHUNK_OVERLAP = 50


def ensure_dirs():
    os.makedirs(RAW_DIR, exist_ok=True)
    os.makedirs(CLEAN_DIR, exist_ok=True)
    os.makedirs(DATA_DIR, exist_ok=True)


def url_key(url: str) -> str:
    return hashlib.sha256(url.encode("utf-8")).hexdigest()


def in_scope(url: str, base_host: str, base_path_prefix: str) -> bool:
    u = urlparse(url)
    return (u.netloc == base_host) and u.path.startswith(base_path_prefix)


def norm_url(base: str, href: str, base_host: str, base_path_prefix: str) -> str | None:
    if not href:
        return None
    u, _ = urldefrag(urljoin(base, href))
    return u if in_scope(u, base_host, base_path_prefix) else None


def fetch_html(url: str) -> str:
    r = requests.get(url, timeout=TIMEOUT_SEC, headers={"User-Agent": "tasamu-rag/1.0"})
    if FORCE_ENCODING:
        r.encoding = FORCE_ENCODING
    else:
        r.encoding = r.apparent_encoding or r.encoding
    return r.text


def html_to_text_and_links(url: str, html: str, base_host: str, base_path_prefix: str):
    soup = BeautifulSoup(html, "html.parser")

    links = []
    for a in soup.select("a[href]"):
        nxt = norm_url(url, a.get("href"), base_host, base_path_prefix)
        if nxt:
            links.append(nxt)

    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    text = soup.get_text(separator="\n")
    text = re.sub(r"\n{3,}", "\n\n", text).strip()
    return text, links


def init_db():
    con = sqlite3.connect(DB_PATH)
    cur = con.cursor()
    cur.execute("""
        CREATE TABLE IF NOT EXISTS pages (
            url TEXT PRIMARY KEY,
            url_key TEXT,
            fetched_at INTEGER,
            content_hash TEXT,
            raw_path TEXT,
            clean_path TEXT,
            status TEXT
        )
    """)
    con.commit()
    return con


def get_page_record(con, url: str):
    cur = con.cursor()
    cur.execute("SELECT content_hash, clean_path FROM pages WHERE url=?", (url,))
    row = cur.fetchone()
    return row  # (content_hash, clean_path) or None


def upsert_page_record(con, url: str, content_hash: str, raw_path: str, clean_path: str, status: str):
    cur = con.cursor()
    cur.execute("""
        INSERT INTO pages(url, url_key, fetched_at, content_hash, raw_path, clean_path, status)
        VALUES(?,?,?,?,?,?,?)
        ON CONFLICT(url) DO UPDATE SET
            fetched_at=excluded.fetched_at,
            content_hash=excluded.content_hash,
            raw_path=excluded.raw_path,
            clean_path=excluded.clean_path,
            status=excluded.status
    """, (url, url_key(url), int(time.time()), content_hash, raw_path, clean_path, status))
    con.commit()


def write_text(path: str, text: str):
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)


def write_raw(path: str, html: str):
    with open(path, "w", encoding="utf-8", errors="ignore") as f:
        f.write(html)


def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def crawl_and_save():
    ensure_dirs()
    con = init_db()

    basep = urlparse(BASE_URL)
    base_host = basep.netloc
    base_path_prefix = basep.path if basep.path.endswith("/") else basep.path + "/"

    q = deque([(BASE_URL, 0)])
    seen = set()
    saved_docs = []

    while q and len(seen) < MAX_PAGES:
        url, depth = q.popleft()
        if url in seen:
            continue
        seen.add(url)

        try:
            html = fetch_html(url)
            text, links = html_to_text_and_links(url, html, base_host, base_path_prefix)

            # 次のURLをキューへ
            if depth < MAX_DEPTH:
                for nxt in links:
                    if nxt not in seen:
                        q.append((nxt, depth + 1))

            # 本文が薄すぎるページは保存しない（必要なら閾値調整）
            if len(text) < 300:
                upsert_page_record(con, url, "", "", "", "skipped_thin")
                time.sleep(SLEEP_SEC)
                continue

            h = sha256_text(text)
            rec = get_page_record(con, url)
            if rec and rec[0] == h:
                # 変更なし：既存cleanを使う
                clean_path = rec[1]
                if clean_path and os.path.exists(clean_path):
                    with open(clean_path, "r", encoding="utf-8") as f:
                        saved_docs.append(Document(page_content=f.read(), metadata={"source": url}))
                upsert_page_record(con, url, h, "", clean_path, "unchanged")
                time.sleep(SLEEP_SEC)
                continue

            # 変更あり/新規：保存
            k = url_key(url)
            raw_path = os.path.join(RAW_DIR, f"{k}.html")
            clean_path = os.path.join(CLEAN_DIR, f"{k}.txt")

            write_raw(raw_path, html)
            write_text(clean_path, text)
            upsert_page_record(con, url, h, raw_path, clean_path, "saved")

            saved_docs.append(Document(page_content=text, metadata={"source": url}))

        except Exception:
            upsert_page_record(con, url, "", "", "", "fetch_failed")

        time.sleep(SLEEP_SEC)

    con.close()
    return saved_docs, len(seen)


def build_faiss(docs):
    splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
    splits = splitter.split_documents(docs)

    db = FAISS.from_documents(splits, OpenAIEmbeddings())
    db.save_local(INDEX_DIR)

    # 参考情報を保存
    meta = {
        "base_url": BASE_URL,
        "chunk_size": CHUNK_SIZE,
        "chunk_overlap": CHUNK_OVERLAP,
        "created_at": int(time.time()),
        "doc_count": len(docs),
        "chunk_count": len(splits),
    }
    with open(os.path.join(DATA_DIR, "index_meta.json"), "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)


def main():
    if not os.environ.get("OPENAI_API_KEY"):
        raise RuntimeError("OPENAI_API_KEY を設定してください")

    docs, visited = crawl_and_save()
    print(f"visited urls: {visited}, loaded docs: {len(docs)}")

    if not docs:
        raise RuntimeError("有効な本文が取得できませんでした。")

    build_faiss(docs)
    print(f"saved index: {INDEX_DIR}")


if __name__ == "__main__":
    main()


Exception ignored in: <function SyncHttpxClientWrapper.__del__ at 0x7d0a28673ba0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/openai/_base_client.py", line 810, in __del__
    def __del__(self) -> None:

KeyboardInterrupt: 


visited urls: 300, loaded docs: 300


In [3]:
!ls -la rag_data/

ls: cannot access 'rag_data/': No such file or directory


In [4]:
import os, glob
print("cwd =", os.getcwd())
print("ls =", os.listdir(".")[:50])
print("glob ./rag_data* =", glob.glob("./rag_data*"))

cwd = /content
ls = ['.config', 'sample_data']
glob ./rag_data* = []


In [9]:
import os

from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAI, OpenAIEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

DATA_DIR  = "/content/drive/MyDrive/rag_store"
INDEX_DIR = os.path.join(DATA_DIR, "index_faiss")

TOP_K = 3
MAX_CTX_CHARS = 4500
MAX_ANSWER_CHARS = 200

emb = OpenAIEmbeddings()
db = FAISS.load_local(INDEX_DIR, emb, allow_dangerous_deserialization=True)
retriever = db.as_retriever(search_kwargs={"k": 6})

def unique_by_source(docs, limit=TOP_K):
    out=[]; seen=set()
    for d in docs:
        s=d.metadata.get("source")
        if s in seen:
            continue
        seen.add(s); out.append(d)
        if len(out)>=limit:
            break
    return out

def format_docs_limited(docs):
    t = "\n\n".join(d.page_content for d in docs)
    return t[:MAX_CTX_CHARS]

prompt = PromptTemplate.from_template(
    "次の情報だけを根拠に質問に答えてください。\n"
    "回答は日本語で200文字以下。\n"
    "情報に無い内容は推測せず「不明」と答えてください。\n\n"
    "情報:\n{context}\n\n質問:\n{question}\n\n回答:"
)

llm = OpenAI(model="gpt-3.5-turbo-instruct", temperature=0, max_tokens=180)

chain = (
    {"context": (retriever | (lambda ds: unique_by_source(ds, TOP_K)) | format_docs_limited),
     "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

def ask(q):
    a = chain.invoke(q).strip().replace("\n"," ")[:MAX_ANSWER_CHARS]
    srcs = [d.metadata.get("source") for d in unique_by_source(retriever.invoke(q), TOP_K)]
    return a, srcs

# 例
q = "海外の大物のでるポイントは？"
a, srcs = ask(q)
print("Q:", q)
print("A:", a)
print("sources:", srcs)

# 対話（やめるときは空Enter）
while True:
    q = input("\nQ> ").strip()
    if not q:
        break
    a, srcs = ask(q)
    print("A>", a)
    print("sources:", srcs)


OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

In [14]:
import os

from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAI, OpenAIEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

DATA_DIR  = "/content/drive/MyDrive/rag_store"
INDEX_DIR = os.path.join(DATA_DIR, "index_faiss")

TOP_K = 3
MAX_CTX_CHARS = 4500
MAX_ANSWER_CHARS = 200

emb = OpenAIEmbeddings()
db = FAISS.load_local(INDEX_DIR, emb, allow_dangerous_deserialization=True)
retriever = db.as_retriever(search_kwargs={"k": 6})

def unique_by_source(docs, limit=TOP_K):
    out=[]; seen=set()
    for d in docs:
        s=d.metadata.get("source")
        if s in seen:
            continue
        seen.add(s); out.append(d)
        if len(out)>=limit:
            break
    return out

def format_docs_limited(docs):
    t = "\n\n".join(d.page_content for d in docs)
    return t[:MAX_CTX_CHARS]

prompt = PromptTemplate.from_template(
    "次の情報だけを根拠に質問に答えてください。\n"
    "回答は日本語で200文字以下。\n"
    "情報に無い内容は推測せず「不明」と答えてください。\n\n"
    "情報:\n{context}\n\n質問:\n{question}\n\n回答:"
)

llm = OpenAI(model="gpt-3.5-turbo-instruct", temperature=0, max_tokens=180)

chain = (
    {"context": (retriever | (lambda ds: unique_by_source(ds, TOP_K)) | format_docs_limited),
     "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

def ask(q):
    a = chain.invoke(q).strip().replace("\n"," ")[:MAX_ANSWER_CHARS]
    srcs = [d.metadata.get("source") for d in unique_by_source(retriever.invoke(q), TOP_K)]
    return a, srcs

# 例
q = "海外の大物のでるポイントは？"
a, srcs = ask(q)
print("Q:", q)
print("A:", a)
print("sources:", srcs)

# 対話（やめるときは空Enter）
while True:
    q = input("\nQ> ").strip()
    if not q:
        break
    a, srcs = ask(q)
    print("A>", a)
    print("sources:", srcs)


RuntimeError: Error in faiss::FileIOReader::FileIOReader(const char*) at /project/third-party/faiss/faiss/impl/io.cpp:69: Error: 'f' failed: could not open /content/drive/MyDrive/rag_store/index_faiss/index.faiss for reading: No such file or directory

In [7]:
# === Colab: RAG（放置→再開OK）フルコード ===
# セルを上から順に実行するだけ

# 0) Drive
from google.colab import drive
drive.mount("/content/drive")

# 1) pip
!pip -q install -U langchain-openai langchain-core langchain-community langchain-text-splitters openai faiss-cpu requests beautifulsoup4

# 2) config
import os, re, time, json, hashlib, sqlite3, glob, requests
from collections import deque
from urllib.parse import urljoin, urldefrag, urlparse
from bs4 import BeautifulSoup

from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAI, OpenAIEmbeddings, OpenAIEmbeddings

BASE_URL = "https://marinediving.com/area/"
DATA_DIR = "/content/drive/MyDrive/rag_store"

RAW_DIR   = os.path.join(DATA_DIR, "raw")
CLEAN_DIR = os.path.join(DATA_DIR, "clean")
DB_PATH   = os.path.join(DATA_DIR, "meta.sqlite3")
INDEX_DIR = os.path.join(DATA_DIR, "index_faiss")

PROGRESS_CRAWL = os.path.join(DATA_DIR, "crawl_progress.json")
PROGRESS_INDEX = os.path.join(DATA_DIR, "index_progress.json")

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(CLEAN_DIR, exist_ok=True)

# crawl params
MAX_DEPTH = 2
MAX_PAGES_PER_RUN = 200
SLEEP_SEC = 0.2
TIMEOUT_SEC = 20

# index params
CHUNK_SIZE = 400
CHUNK_OVERLAP = 50
BATCH_FILES = 50
BATCH_SPLITS = 200
MAX_FILES_PER_RUN = 200

# ask params
TOP_K = 3
MAX_CTX_CHARS = 4500
MAX_ANSWER_CHARS = 200

# 3) API key
import getpass
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key: ")

# 4) crawl (resume)
basep = urlparse(BASE_URL)
BASE_HOST = basep.netloc
BASE_PATH_PREFIX = basep.path if basep.path.endswith("/") else basep.path + "/"

def url_key(url: str) -> str:
    return hashlib.sha256(url.encode("utf-8")).hexdigest()

def in_scope(url: str) -> bool:
    u = urlparse(url)
    return (u.netloc == BASE_HOST) and u.path.startswith(BASE_PATH_PREFIX)

def norm_url(base: str, href: str):
    if not href:
        return None
    u, _ = urldefrag(urljoin(base, href))
    return u if in_scope(u) else None

def fetch_html(url: str) -> str:
    r = requests.get(url, timeout=TIMEOUT_SEC, headers={"User-Agent": "tasamu-rag/1.0"})
    r.encoding = r.apparent_encoding or r.encoding
    return r.text

def html_to_text_and_links(url: str, html: str):
    soup = BeautifulSoup(html, "html.parser")
    links = []
    for a in soup.select("a[href]"):
        nxt = norm_url(url, a.get("href"))
        if nxt:
            links.append(nxt)
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()
    text = soup.get_text(separator="\n")
    text = re.sub(r"\n{3,}", "\n\n", text).strip()
    return text, links

def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

con = sqlite3.connect(DB_PATH)
cur = con.cursor()
cur.execute("""
CREATE TABLE IF NOT EXISTS pages(
  url TEXT PRIMARY KEY,
  url_key TEXT,
  fetched_at INTEGER,
  content_hash TEXT,
  clean_path TEXT,
  status TEXT
)
""")
con.commit()

def get_record(url: str):
    cur = con.cursor()
    cur.execute("SELECT status, content_hash, clean_path FROM pages WHERE url=?", (url,))
    return cur.fetchone()

def upsert(url: str, content_hash: str, clean_path: str, status: str):
    cur = con.cursor()
    cur.execute("""
    INSERT INTO pages(url, url_key, fetched_at, content_hash, clean_path, status)
    VALUES(?,?,?,?,?,?)
    ON CONFLICT(url) DO UPDATE SET
      fetched_at=excluded.fetched_at,
      content_hash=excluded.content_hash,
      clean_path=excluded.clean_path,
      status=excluded.status
    """, (url, url_key(url), int(time.time()), content_hash, clean_path, status))
    con.commit()

if os.path.exists(PROGRESS_CRAWL):
    with open(PROGRESS_CRAWL, "r", encoding="utf-8") as f:
        st = json.load(f)
    q = deque(st.get("queue", []))
    seen = set(st.get("seen", []))
    processed_total = st.get("processed_total", 0)
else:
    q = deque([(BASE_URL, 0)])
    seen = set()
    processed_total = 0

processed_this_run = 0
saved_this_run = 0

while q and processed_this_run < MAX_PAGES_PER_RUN:
    url, depth = q.popleft()
    if url in seen:
        continue
    seen.add(url)

    rec = get_record(url)
    if rec and rec[0] in ("saved", "unchanged", "skipped_thin"):
        processed_this_run += 1
        processed_total += 1
        continue

    try:
        html = fetch_html(url)
        text, links = html_to_text_and_links(url, html)

        if depth < MAX_DEPTH:
            for nxt in links:
                if nxt not in seen:
                    q.append((nxt, depth + 1))

        if len(text) < 300:
            upsert(url, "", "", "skipped_thin")
        else:
            h = sha256_text(text)
            k = url_key(url)
            clean_path = os.path.join(CLEAN_DIR, f"{k}.txt")
            with open(clean_path, "w", encoding="utf-8") as f:
                f.write(text)
            upsert(url, h, clean_path, "saved")
            saved_this_run += 1

    except Exception:
        upsert(url, "", "", "fetch_failed")

    processed_this_run += 1
    processed_total += 1
    time.sleep(SLEEP_SEC)

with open(PROGRESS_CRAWL, "w", encoding="utf-8") as f:
    json.dump({"queue": list(q), "seen": list(seen), "processed_total": processed_total}, f, ensure_ascii=False)

con.close()

print("crawl processed:", processed_this_run, "saved:", saved_this_run, "queue:", len(q), "clean_files:", len(glob.glob(os.path.join(CLEAN_DIR, '*.txt'))))

# 5) index (resume)
emb = OpenAIEmbeddings()
splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)

files = sorted(glob.glob(os.path.join(CLEAN_DIR, "*.txt")))
start_idx = 0
if os.path.exists(PROGRESS_INDEX):
    with open(PROGRESS_INDEX, "r", encoding="utf-8") as f:
        start_idx = json.load(f).get("next_file_idx", 0)

db = None
if os.path.isdir(INDEX_DIR):
    try:
        db = FAISS.load_local(INDEX_DIR, emb, allow_dangerous_deserialization=True)
    except Exception:
        db = None

end_limit = min(len(files), start_idx + MAX_FILES_PER_RUN)

for fi in range(start_idx, end_limit, BATCH_FILES):
    batch_files = files[fi:fi+BATCH_FILES]
    docs = []
    for p in batch_files:
        with open(p, "r", encoding="utf-8") as f:
            txt = f.read()
        docs.append(Document(page_content=txt, metadata={"source": os.path.basename(p)}))

    splits = splitter.split_documents(docs)

    for si in range(0, len(splits), BATCH_SPLITS):
        part = splits[si:si+BATCH_SPLITS]
        if db is None:
            db = FAISS.from_documents(part, emb)
        else:
            db.add_documents(part)
        os.makedirs(INDEX_DIR, exist_ok=True)
        db.save_local(INDEX_DIR)

    with open(PROGRESS_INDEX, "w", encoding="utf-8") as f:
        json.dump({"next_file_idx": fi + len(batch_files), "total_files": len(files)}, f, ensure_ascii=False)

print("index_dir:", INDEX_DIR)

# 6) ask (no site access)
retriever = db.as_retriever(search_kwargs={"k": 6})
prompt = PromptTemplate.from_template(
    "次の情報だけを根拠に質問に答えてください。\n"
    "回答は日本語で200文字以下。\n"
    "情報に無い内容は推測せず「不明」と答えてください。\n\n"
    "情報:\n{context}\n\n質問:\n{question}\n\n回答:"
)
llm = OpenAI(model="gpt-3.5-turbo-instruct", temperature=0, max_tokens=180)

from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def unique_by_source(docs, limit=TOP_K):
    out=[]; s=set()
    for d in docs:
        src=d.metadata.get("source")
        if src in s:
            continue
        s.add(src); out.append(d)
        if len(out)>=limit:
            break
    return out

def format_docs_limited(docs):
    t="\n\n".join(d.page_content for d in docs)
    return t[:MAX_CTX_CHARS]

chain = (
    {"context": (retriever | (lambda ds: unique_by_source(ds, TOP_K)) | format_docs_limited),
     "question": RunnablePassthrough()}
    | prompt | llm | StrOutputParser()
)

q = "海外の大物のでるポイントは？"
a = chain.invoke(q).strip().replace("\n", " ")[:MAX_ANSWER_CHARS]
print("Q:", q)
print("A:", a)


Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.7/107.7 kB 3.3 MB/s eta 0:00:00


OpenAI API Key: ··········
crawl processed: 135 saved: 135 queue: 0 clean_files: 335
index_dir: /content/drive/MyDrive/rag_store/index_faiss
Q: 海外の大物のでるポイントは？
A: 情報には海外の情報は含まれていないため、不明となります。


In [3]:
!pip install -U langchain-openai langchain-core langchain-community langchain-text-splitters openai faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.8/84.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 490.2/490.2 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 85.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: openai
    Found existing installation: openai 2.14.0
    Uninstalling openai-2.14.0:
      Successfully uninstalled openai-2.14.0
  Attempting uninstall: langchain-core
    Found existing

In [11]:
# 追加セル：全件に近づける（深さは固定、上限だけ解除/反復）
# 使い方：
# 1) 下の SETTINGS を好みに変更
# 2) このセルを実行（終わるまで放置OK）
# 3) index が増えたら質問セルを実行

import os, re, time, json, hashlib, sqlite3, glob, requests
from collections import deque
from urllib.parse import urljoin, urldefrag, urlparse
from bs4 import BeautifulSoup

from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings

# === SETTINGS ===
BASE_URL = "https://marinediving.com/area/"
DATA_DIR = "/content/drive/MyDrive/rag_store"

MAX_DEPTH = 2               # リンク深さは固定
RUN_CRAWL_LIMIT = 9999999   # 1回で進める上限（事実上全件）
RUN_INDEX_LIMIT = 9999999   # 1回で進める上限（事実上全件）
STOP_IF_QUEUE_EMPTY = True  # キューが空なら終了

SLEEP_SEC = 0.2
TIMEOUT_SEC = 20

# index params
CHUNK_SIZE = 400
CHUNK_OVERLAP = 50
BATCH_FILES = 50
BATCH_SPLITS = 200

RAW_DIR   = os.path.join(DATA_DIR, "raw")
CLEAN_DIR = os.path.join(DATA_DIR, "clean")
DB_PATH   = os.path.join(DATA_DIR, "meta.sqlite3")
INDEX_DIR = os.path.join(DATA_DIR, "index_faiss")
PROGRESS_CRAWL = os.path.join(DATA_DIR, "crawl_progress.json")
PROGRESS_INDEX = os.path.join(DATA_DIR, "index_progress.json")

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(CLEAN_DIR, exist_ok=True)
os.makedirs(INDEX_DIR, exist_ok=True)

basep = urlparse(BASE_URL)
BASE_HOST = basep.netloc
BASE_PATH_PREFIX = basep.path if basep.path.endswith("/") else basep.path + "/"

def url_key(url: str) -> str:
    return hashlib.sha256(url.encode("utf-8")).hexdigest()

def in_scope(url: str) -> bool:
    u = urlparse(url)
    return (u.netloc == BASE_HOST) and u.path.startswith(BASE_PATH_PREFIX)

def norm_url(base: str, href: str):
    if not href:
        return None
    u, _ = urldefrag(urljoin(base, href))
    return u if in_scope(u) else None

def fetch_html(url: str) -> str:
    r = requests.get(url, timeout=TIMEOUT_SEC, headers={"User-Agent":"tasamu-rag/1.0"})
    r.encoding = r.apparent_encoding or r.encoding
    return r.text

def html_to_text_and_links(url: str, html: str):
    soup = BeautifulSoup(html, "html.parser")
    links = []
    for a in soup.select("a[href]"):
        nxt = norm_url(url, a.get("href"))
        if nxt:
            links.append(nxt)
    for tag in soup(["script","style","noscript"]):
        tag.decompose()
    text = soup.get_text("\n")
    text = re.sub(r"\n{3,}","\n\n",text).strip()
    return text, links

def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

# --- DB ---
con = sqlite3.connect(DB_PATH)
cur = con.cursor()
cur.execute("""
CREATE TABLE IF NOT EXISTS pages(
  url TEXT PRIMARY KEY,
  url_key TEXT,
  fetched_at INTEGER,
  content_hash TEXT,
  clean_path TEXT,
  status TEXT
)
""")
con.commit()

def get_record(url: str):
    cur = con.cursor()
    cur.execute("SELECT status, content_hash, clean_path FROM pages WHERE url=?", (url,))
    return cur.fetchone()

def upsert(url: str, content_hash: str, clean_path: str, status: str):
    cur = con.cursor()
    cur.execute("""
    INSERT INTO pages(url, url_key, fetched_at, content_hash, clean_path, status)
    VALUES(?,?,?,?,?,?)
    ON CONFLICT(url) DO UPDATE SET
      fetched_at=excluded.fetched_at,
      content_hash=excluded.content_hash,
      clean_path=excluded.clean_path,
      status=excluded.status
    """, (url, url_key(url), int(time.time()), content_hash, clean_path, status))
    con.commit()

# --- Crawl resume state ---
if os.path.exists(PROGRESS_CRAWL):
    with open(PROGRESS_CRAWL, "r", encoding="utf-8") as f:
        st = json.load(f)
    q = deque(st.get("queue", []))
    seen = set(st.get("seen", []))
    processed_total = st.get("processed_total", 0)
else:
    q = deque([(BASE_URL, 0)])
    seen = set()
    processed_total = 0

processed_this_run = 0
saved_this_run = 0

while q and processed_this_run < RUN_CRAWL_LIMIT:
    url, depth = q.popleft()
    if url in seen:
        continue
    seen.add(url)

    rec = get_record(url)
    if rec and rec[0] in ("saved","unchanged","skipped_thin"):
        processed_this_run += 1
        processed_total += 1
        continue

    try:
        html = fetch_html(url)
        text, links = html_to_text_and_links(url, html)

        if depth < MAX_DEPTH:
            for nxt in links:
                if nxt not in seen:
                    q.append((nxt, depth + 1))

        if len(text) < 300:
            upsert(url, "", "", "skipped_thin")
        else:
            h = sha256_text(text)
            k = url_key(url)
            clean_path = os.path.join(CLEAN_DIR, f"{k}.txt")
            with open(clean_path, "w", encoding="utf-8") as f:
                f.write(text)
            upsert(url, h, clean_path, "saved")
            saved_this_run += 1

    except Exception:
        upsert(url, "", "", "fetch_failed")

    processed_this_run += 1
    processed_total += 1
    if processed_this_run % 200 == 0:
        print("crawl processed:", processed_this_run, "saved:", saved_this_run, "queue:", len(q))
    time.sleep(SLEEP_SEC)

with open(PROGRESS_CRAWL, "w", encoding="utf-8") as f:
    json.dump({"queue": list(q), "seen": list(seen), "processed_total": processed_total}, f, ensure_ascii=False)

con.close()

clean_files = len(glob.glob(os.path.join(CLEAN_DIR, "*.txt")))
print("crawl processed:", processed_this_run, "saved:", saved_this_run, "queue:", len(q), "clean_files:", clean_files)

# --- Index resume ---
emb = OpenAIEmbeddings()
splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)

files = sorted(glob.glob(os.path.join(CLEAN_DIR, "*.txt")))
start_idx = 0
if os.path.exists(PROGRESS_INDEX):
    with open(PROGRESS_INDEX, "r", encoding="utf-8") as f:
        start_idx = json.load(f).get("next_file_idx", 0)

db = None
if os.path.isdir(INDEX_DIR):
    try:
        db = FAISS.load_local(INDEX_DIR, emb, allow_dangerous_deserialization=True)
    except Exception:
        db = None

end_limit = min(len(files), start_idx + RUN_INDEX_LIMIT)
for fi in range(start_idx, end_limit, BATCH_FILES):
    batch_files = files[fi:fi+BATCH_FILES]
    docs = []
    for p in batch_files:
        with open(p, "r", encoding="utf-8") as f:
            txt = f.read()
        docs.append(Document(page_content=txt, metadata={"source": p}))

    splits = splitter.split_documents(docs)

    for si in range(0, len(splits), BATCH_SPLITS):
        part = splits[si:si+BATCH_SPLITS]
        if db is None:
            db = FAISS.from_documents(part, emb)
        else:
            db.add_documents(part)
        db.save_local(INDEX_DIR)

    with open(PROGRESS_INDEX, "w", encoding="utf-8") as f:
        json.dump({"next_file_idx": fi + len(batch_files), "total_files": len(files)}, f, ensure_ascii=False)

    if (fi - start_idx) % (BATCH_FILES*4) == 0:
        print("index files:", fi + len(batch_files), "/", len(files))

print("index_dir:", INDEX_DIR)
print("progress_index:", PROGRESS_INDEX)
print("queue_remaining:", len(q))


crawl processed: 0 saved: 0 queue: 0 clean_files: 335
index_dir: /content/drive/MyDrive/rag_store/index_faiss
progress_index: /content/drive/MyDrive/rag_store/index_progress.json
queue_remaining: 0


In [13]:
!ls -l /content/drive/MyDrive/rag_store/index_faiss

total 49788
-rw-r--r-- 1 root root 43634733 Jan 12 12:57 index.faiss
-rw-r--r-- 1 root root  7341828 Jan 12 12:57 index.pkl


In [9]:
!pip -q install -U langchain-core langchain-openai langchain-community langchain-text-splitters openai faiss-cpu requests beautifulsoup4


In [8]:
# Colab: 質問（RAG）セル
import os
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAI, OpenAIEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

DATA_DIR  = "/content/drive/MyDrive/rag_store"
INDEX_DIR = os.path.join(DATA_DIR, "index_faiss")

TOP_K = 3
MAX_CTX_CHARS = 4500
MAX_ANSWER_CHARS = 200

emb = OpenAIEmbeddings()
db = FAISS.load_local(INDEX_DIR, emb, allow_dangerous_deserialization=True)
retriever = db.as_retriever(search_kwargs={"k": 10})

def unique_by_source(docs, limit=TOP_K):
    out=[]; seen=set()
    for d in docs:
        s = d.metadata.get("source")
        if s in seen:
            continue
        seen.add(s); out.append(d)
        if len(out) >= limit:
            break
    return out

def format_docs_limited(docs):
    t = "\n\n".join(d.page_content for d in docs)
    return t[:MAX_CTX_CHARS]

prompt = PromptTemplate.from_template(
    "次の情報だけを根拠に質問に答えてください。\n"
    "回答は日本語で200文字以下。\n"
    "情報に無い内容は推測せず「不明」と答えてください。\n\n"
    "情報:\n{context}\n\n質問:\n{question}\n\n回答:"
)

llm = OpenAI(model="gpt-3.5-turbo-instruct", temperature=0, max_tokens=180)

chain = (
    {"context": (retriever | (lambda ds: unique_by_source(ds, TOP_K)) | format_docs_limited),
     "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

def ask(q: str):
    a = chain.invoke(q).strip().replace("\n", " ")[:MAX_ANSWER_CHARS]
    srcs = [d.metadata.get("source") for d in unique_by_source(retriever.invoke(q), TOP_K)]
    return a, srcs

# 例
q = "海外の大物のでるポイントは？"
a, srcs = ask(q)
print("Q:", q)
print("A:", a)
print("sources:", srcs)

# 対話（空Enterで終了）
while True:
    q = input("\nQ> ").strip()
    if not q:
        break
    a, srcs = ask(q)
    print("A>", a)
    print("sources:", srcs)


Q: 海外の大物のでるポイントは？
A: 情報には海外の情報は含まれていないため、不明となります。
sources: ['025e27754c8ba50cb39a8b8005032d17d62ae4b2e59fee4ed6b31e70c4f078a3.txt', 'a1155aa4b9e065c39176abd75ecb0489248e53db86d01c74418d86fb01dbf16d.txt', 'd25827eebc98136534345ec7836030be38bd09c3cb93bef3fa94343db3d68457.txt']

Q> ハンマーの出るポイントは
A> カメ根、ジャブ根、Aポイントの3カ所が主なポイントとして知られています。また、遺跡ポイントや西崎の沖合でもハンマーを見ることができる可能性があります。
sources: ['57c27f794f1ede725f9a28d0a19aa824967e048731bf9dc771f25aa6c3afe986.txt', 'f2c6fd56780e735c96f4fa4847cdf25d92497e63ec35c18f648140660b27b844.txt', 'ba300809fcb8a7e16cffe2511091a5c9c6b26b03b3a8c7c2e95bb8fb22f6e79c.txt']

Q> それはどこですか？
A> 柏島は、四国の南西端に位置する高知県大月町にあります。
sources: ['3673e3841f7350ef87620b0b6de0b432d042685544b4f420b3951d551b7ba3f8.txt', '5e276414a9b64a97e5a22d3095f07ad3a418b524576f840de7927599fe1ac664.txt', 'a21832837944319fd679ce5ddff571996008a6ca14807bab6e67804d0245c59d.txt']

Q> 栢島でハンマーヘッドが出るのですか？
A> 不明。栢島は情報に記載されていないため、ハンマーヘッドが出るかどうかはわかりません。ただし、与那国島や神子元島のように黒潮が流れる海域であれば、ハンマーヘッドが出る可能性は高いと考えられます。また、潮の動きや透

KeyboardInterrupt: Interrupted by user

In [9]:
# ===== Colab: marinediving /area/ 全ページスキャン（depth制限あり）+ RAG(FAISS) 作成 + 質問 =====
# 使い方：上から順に実行。切断しても Drive の進捗で再開します。

# --- 0) Drive ---
from google.colab import drive
drive.mount("/content/drive")

# --- 1) pip ---
!pip -q install -U langchain-openai langchain-core langchain-community langchain-text-splitters openai faiss-cpu requests beautifulsoup4

# --- 2) 設定 ---
import os, re, time, json, hashlib, sqlite3, glob, requests
from collections import deque
from urllib.parse import urljoin, urldefrag, urlparse
from bs4 import BeautifulSoup

from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, OpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

BASE_URL = "https://marinediving.com/area/"          # 固定
DATA_DIR = "/content/drive/MyDrive/rag_store_md"     # 保存先（Drive）

# depth制限（要望の通り）
MAX_DEPTH = 2

# 1回の実行で進める上限（大きくすると長時間になりColabで切れやすい）
# 「全ページスキャン」は、これを何回か回して queue を 0 にします
MAX_PAGES_PER_RUN = 500

SLEEP_SEC = 0.15
TIMEOUT_SEC = 20

# index
CHUNK_SIZE = 900
CHUNK_OVERLAP = 120
BATCH_FILES = 50
BATCH_SPLITS = 200
MAX_FILES_PER_RUN = 9999999   # 可能なら一気に（重ければ 200 などに）

# ask
TOP_K = 3
MAX_CTX_CHARS = 5200
MAX_ANSWER_CHARS = 200

RAW_DIR   = os.path.join(DATA_DIR, "raw")
CLEAN_DIR = os.path.join(DATA_DIR, "clean")
DB_PATH   = os.path.join(DATA_DIR, "meta.sqlite3")
INDEX_DIR = os.path.join(DATA_DIR, "index_faiss")

PROGRESS_CRAWL = os.path.join(DATA_DIR, "crawl_progress.json")
PROGRESS_INDEX = os.path.join(DATA_DIR, "index_progress.json")

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(CLEAN_DIR, exist_ok=True)
os.makedirs(INDEX_DIR, exist_ok=True)

# --- 3) APIキー（毎回） ---
import getpass
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key: ")

# --- 4) クロール（/area/配下のみ + depth制限 + 全ページに近づける） ---
basep = urlparse(BASE_URL)
BASE_HOST = basep.netloc
BASE_PATH_PREFIX = basep.path if basep.path.endswith("/") else basep.path + "/"

def url_key(url: str) -> str:
    return hashlib.sha256(url.encode("utf-8")).hexdigest()

def in_scope(url: str) -> bool:
    u = urlparse(url)
    return (u.netloc == BASE_HOST) and u.path.startswith(BASE_PATH_PREFIX)

def norm_url(base: str, href: str):
    if not href:
        return None
    u, _ = urldefrag(urljoin(base, href))
    return u if in_scope(u) else None

def fetch_html(url: str) -> str:
    r = requests.get(url, timeout=TIMEOUT_SEC, headers={"User-Agent": "tasamu-rag/1.0"})
    r.encoding = r.apparent_encoding or r.encoding
    return r.text

def html_to_text_and_links(url: str, html: str):
    soup = BeautifulSoup(html, "html.parser")

    # links
    links = []
    for a in soup.select("a[href]"):
        nxt = norm_url(url, a.get("href"))
        if nxt:
            links.append(nxt)

    # noise drop
    for tag in soup(["script", "style", "noscript", "header", "footer", "nav", "aside"]):
        tag.decompose()

    # main/article優先
    main = (soup.select_one("main")
            or soup.select_one("article")
            or soup.select_one("div.entry-content")
            or soup.select_one("div.content")
            or soup.body
            or soup)

    title = soup.title.get_text(" ", strip=True) if soup.title else ""

    text = main.get_text("\n")
    text = re.sub(r"\n{3,}", "\n\n", text).strip()

    if title:
        text = title + "\n\n" + text

    return text, links

def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

# sqlite
con = sqlite3.connect(DB_PATH)
cur = con.cursor()
cur.execute("""
CREATE TABLE IF NOT EXISTS pages(
  url TEXT PRIMARY KEY,
  url_key TEXT,
  fetched_at INTEGER,
  content_hash TEXT,
  clean_path TEXT,
  status TEXT
)
""")
con.commit()

def get_record(url: str):
    cur = con.cursor()
    cur.execute("SELECT status, content_hash, clean_path FROM pages WHERE url=?", (url,))
    return cur.fetchone()

def upsert(url: str, content_hash: str, clean_path: str, status: str):
    cur = con.cursor()
    cur.execute("""
    INSERT INTO pages(url, url_key, fetched_at, content_hash, clean_path, status)
    VALUES(?,?,?,?,?,?)
    ON CONFLICT(url) DO UPDATE SET
      fetched_at=excluded.fetched_at,
      content_hash=excluded.content_hash,
      clean_path=excluded.clean_path,
      status=excluded.status
    """, (url, url_key(url), int(time.time()), content_hash, clean_path, status))
    con.commit()

# resume queue/seen
if os.path.exists(PROGRESS_CRAWL):
    with open(PROGRESS_CRAWL, "r", encoding="utf-8") as f:
        st = json.load(f)
    q = deque(st.get("queue", []))
    seen = set(st.get("seen", []))
    processed_total = st.get("processed_total", 0)
else:
    q = deque([(BASE_URL, 0)])
    seen = set()
    processed_total = 0

processed_this_run = 0
saved_this_run = 0

while q and processed_this_run < MAX_PAGES_PER_RUN:
    url, depth = q.popleft()
    if url in seen:
        continue
    seen.add(url)

    rec = get_record(url)
    if rec and rec[0] in ("saved", "unchanged", "skipped_thin"):
        processed_this_run += 1
        processed_total += 1
        continue

    try:
        html = fetch_html(url)
        text, links = html_to_text_and_links(url, html)

        # depth制限
        if depth < MAX_DEPTH:
            for nxt in links:
                if nxt not in seen:
                    q.append((nxt, depth + 1))

        if len(text) < 300:
            upsert(url, "", "", "skipped_thin")
        else:
            h = sha256_text(text)
            k = url_key(url)
            clean_path = os.path.join(CLEAN_DIR, f"{k}.txt")
            with open(clean_path, "w", encoding="utf-8") as f:
                f.write(text)
            upsert(url, h, clean_path, "saved")
            saved_this_run += 1

    except Exception:
        upsert(url, "", "", "fetch_failed")

    processed_this_run += 1
    processed_total += 1
    time.sleep(SLEEP_SEC)

with open(PROGRESS_CRAWL, "w", encoding="utf-8") as f:
    json.dump({"queue": list(q), "seen": list(seen), "processed_total": processed_total}, f, ensure_ascii=False)

con.close()

clean_files = len(glob.glob(os.path.join(CLEAN_DIR, "*.txt")))
print("crawl processed:", processed_this_run, "saved:", saved_this_run, "queue:", len(q), "clean_files:", clean_files)

# --- 5) インデックス作成（FAISS） resume ---
emb = OpenAIEmbeddings()
splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)

files = sorted(glob.glob(os.path.join(CLEAN_DIR, "*.txt")))
start_idx = 0
if os.path.exists(PROGRESS_INDEX):
    with open(PROGRESS_INDEX, "r", encoding="utf-8") as f:
        start_idx = json.load(f).get("next_file_idx", 0)

db = None
if os.path.isdir(INDEX_DIR):
    try:
        db = FAISS.load_local(INDEX_DIR, emb, allow_dangerous_deserialization=True)
    except Exception:
        db = None

end_limit = min(len(files), start_idx + MAX_FILES_PER_RUN)
for fi in range(start_idx, end_limit, BATCH_FILES):
    batch_files = files[fi:fi+BATCH_FILES]
    docs = []
    for p in batch_files:
        with open(p, "r", encoding="utf-8") as f:
            txt = f.read()
        docs.append(Document(page_content=txt, metadata={"source": p}))

    splits = splitter.split_documents(docs)

    for si in range(0, len(splits), BATCH_SPLITS):
        part = splits[si:si+BATCH_SPLITS]
        if db is None:
            db = FAISS.from_documents(part, emb)
        else:
            db.add_documents(part)
        db.save_local(INDEX_DIR)

    with open(PROGRESS_INDEX, "w", encoding="utf-8") as f:
        json.dump({"next_file_idx": fi + len(batch_files), "total_files": len(files)}, f, ensure_ascii=False)

print("index_dir:", INDEX_DIR)
print("progress_index:", PROGRESS_INDEX)
print("queue_remaining:", len(q))

# --- 6) 質問（RAG） ---
retriever = db.as_retriever(search_kwargs={"k": 10})

def unique_by_source(docs, limit=TOP_K):
    out=[]; seen=set()
    for d in docs:
        src = d.metadata.get("source")
        if src in seen:
            continue
        seen.add(src); out.append(d)
        if len(out) >= limit:
            break
    return out

def format_docs_limited(docs):
    t = "\n\n".join(d.page_content for d in docs)
    return t[:MAX_CTX_CHARS]

prompt = PromptTemplate.from_template(
    "次の情報だけを根拠に質問に答えてください。\n"
    "回答は日本語で200文字以下。\n"
    "情報に無い内容は推測せず「記載なし」と答えてください。\n\n"
    "情報:\n{context}\n\n質問:\n{question}\n\n回答:"
)

llm = OpenAI(model="gpt-3.5-turbo-instruct", temperature=0, max_tokens=180)

chain = (
    {"context": (retriever | (lambda ds: unique_by_source(ds, TOP_K)) | format_docs_limited),
     "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

q = "海外の大物のでるポイントは？"
a = chain.invoke(q).strip().replace("\n"," ")[:MAX_ANSWER_CHARS]
print("Q:", q)
print("A:", a)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
OpenAI API Key: ··········
crawl processed: 335 saved: 253 queue: 0 clean_files: 253
index_dir: /content/drive/MyDrive/rag_store_md/index_faiss
progress_index: /content/drive/MyDrive/rag_store_md/index_progress.json
queue_remaining: 0
Q: 海外の大物のでるポイントは？
A: 情報には記載がありませんが、外洋側や岩場などで大物が出る可能性が高いと推測されます。また、サイパンのようにサメの多いポイントも大物が出る可能性が高いと言われています。


In [ ]:
# 質問セル（marinediving /area/ のRAGインデックスが既に作成済み前提）
import os
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAI, OpenAIEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

DATA_DIR  = "/content/drive/MyDrive/rag_store_md"
INDEX_DIR = os.path.join(DATA_DIR, "index_faiss")

TOP_K = 3
MAX_CTX_CHARS = 5200
MAX_ANSWER_CHARS = 200

emb = OpenAIEmbeddings()
db = FAISS.load_local(INDEX_DIR, emb, allow_dangerous_deserialization=True)
retriever = db.as_retriever(search_kwargs={"k": 10})

def unique_by_source(docs, limit=TOP_K):
    out=[]; seen=set()
    for d in docs:
        s = d.metadata.get("source")
        if s in seen:
            continue
        seen.add(s); out.append(d)
        if len(out) >= limit:
            break
    return out

def format_docs_limited(docs):
    t = "\n\n".join(d.page_content for d in docs)
    return t[:MAX_CTX_CHARS]

prompt = PromptTemplate.from_template(
    "次の情報だけを根拠に質問に答えてください。\n"
    "回答は日本語で200文字以下。\n"
    "情報に無い内容は推測せず「記載なし」と答えてください。\n\n"
    "情報:\n{context}\n\n質問:\n{question}\n\n回答:"
)

llm = OpenAI(model="gpt-3.5-turbo-instruct", temperature=0, max_tokens=180)

chain = (
    {"context": (retriever | (lambda ds: unique_by_source(ds, TOP_K)) | format_docs_limited),
     "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

def ask(q: str):
    a = chain.invoke(q).strip().replace("\n", " ")[:MAX_ANSWER_CHARS]
    srcs = [d.metadata.get("source") for d in unique_by_source(retriever.invoke(q), TOP_K)]
    return a, srcs

# 例
q = "海外の大物のでるポイントは？"
a, srcs = ask(q)
print("Q:", q)
print("A:", a)
print("sources:", srcs)

# 対話（空Enterで終了）
while True:
    q = input("\nQ> ").strip()
    if not q:
        break
    a, srcs = ask(q)
    print("A>", a)
    print("sources:", srcs)


Q: 海外の大物のでるポイントは？
A: 情報には記載がありませんが、外洋側や岩場などで大物が出る可能性が高いと推測されます。また、サイパンのようにサメの多いポイントも大物が出る可能性が高いと言われています。
sources: ['/content/drive/MyDrive/rag_store_md/clean/d25827eebc98136534345ec7836030be38bd09c3cb93bef3fa94343db3d68457.txt', '/content/drive/MyDrive/rag_store_md/clean/0ade7e1a6301e2b454df83485563f3ce61453884ec860c1efc1b36d30900b2a6.txt', '/content/drive/MyDrive/rag_store_md/clean/40abc81e4de72d0ea48d3d589e3bec98f0900b09a488b8ce363d9530a10837b1.txt']
A> ハンマーヘッドの出るポイントは、与那国島の南側にあるカメ根、ジャブ根、Aポイントの3カ所が主なポイントとして知られています。特にカメ根やジャブ根からのドリフトコースは、ハンマーが見られる確率が高いと言われています。また、西崎沖でもハンマーヘッドの大群が見られることがあります。シーズンは夏から秋が最も狙い目で、冬から春にかけても見られることがあります
sources: ['/content/drive/MyDrive/rag_store_md/clean/f2c6fd56780e735c96f4fa4847cdf25d92497e63ec35c18f648140660b27b844.txt', '/content/drive/MyDrive/rag_store_md/clean/ba300809fcb8a7e16cffe2511091a5c9c6b26b03b3a8c7c2e95bb8fb22f6e79c.txt', '/content/drive/MyDrive/rag_store_md/clean/bb09996b9057cd9ebf4b9bd1f5876f90b697cea7ee0a799b03a7819472ff2683.txt']
A> 神子元島。神子元島